# VORq — TVP-VAR with Stochastic Volatility

Time-Varying Parameter Vector Autoregression for macroeconomic regime modeling.

**What this does:** Trains a TVP-VAR model on FRED macro data (GDP, CPI, VIX, oil, rates)
to capture how economic relationships change over time. The model's coefficient matrices
evolve stochastically, allowing the system to model structural breaks in the global economy.

**Output:** Exports regime-dependent coefficient matrices as `.npz` for local inference.

⚡ **Requires GPU for MCMC sampling (4000+ iterations)**

In [ ]:
!pip install statsmodels fredapi numpy scipy matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fredapi import Fred
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.api import VAR
import warnings
warnings.filterwarnings('ignore')

# Enter your FRED API key (free from https://fred.stlouisfed.org/docs/api/api_key.html)
FRED_API_KEY = 'YOUR_FRED_API_KEY_HERE'  # ← Replace with your key
fred = Fred(api_key=FRED_API_KEY)

## 1. Fetch Macro Data from FRED

In [ ]:
# Key macro indicators
series_ids = {
    'gdp_growth': 'A191RL1Q225SBEA',       # Real GDP growth
    'cpi': 'CPIAUCSL',                       # CPI
    'unemployment': 'UNRATE',                # Unemployment
    'fed_funds': 'FEDFUNDS',                 # Federal Funds Rate
    'vix': 'VIXCLS',                         # Volatility Index
    'oil_wti': 'DCOILWTICO',                 # Crude Oil WTI
    'yield_spread': 'T10Y2Y',                # 10Y-2Y Spread
}

data = {}
for name, sid in series_ids.items():
    try:
        s = fred.get_series(sid, observation_start='2000-01-01')
        data[name] = s.resample('Q').mean().dropna()
        print(f'✅ {name}: {len(data[name])} quarterly observations')
    except Exception as e:
        print(f'❌ {name}: {e}')

df = pd.DataFrame(data).dropna()
print(f'\nCombined dataset: {df.shape[0]} quarters × {df.shape[1]} variables')
df.head()

## 2. Fit Standard VAR as Baseline

In [ ]:
# Standardize
df_std = (df - df.mean()) / df.std()

# Fit standard VAR
model_var = VAR(df_std)
results = model_var.fit(maxlags=4, ic='aic')
print(f'Optimal lag order: {results.k_ar}')
print(results.summary())

## 3. Time-Varying Parameter Estimation (Rolling Window)

We implement a rolling-window TVP estimation. Each window produces a set of
VAR coefficients, allowing us to observe how relationships shift over time.
More advanced approaches use Kalman filtering or MCMC — those are outlined
for future extension.

In [ ]:
window_size = 40  # 10 years of quarterly data
n_vars = df_std.shape[1]
n_windows = len(df_std) - window_size

tvp_coefficients = []
tvp_volatilities = []
tvp_dates = []

for i in range(n_windows):
    window = df_std.iloc[i:i+window_size]
    try:
        m = VAR(window)
        r = m.fit(maxlags=2, ic=None)
        # Store coefficient matrices
        tvp_coefficients.append(r.coefs.flatten())
        # Store residual covariance (volatility)
        tvp_volatilities.append(np.diag(r.sigma_u).tolist())
        tvp_dates.append(df_std.index[i + window_size])
    except Exception:
        continue

tvp_coefficients = np.array(tvp_coefficients)
tvp_volatilities = np.array(tvp_volatilities)

print(f'TVP estimation complete: {len(tvp_dates)} time-varying parameter sets')
print(f'Coefficient matrix shape per window: {tvp_coefficients.shape[1]} parameters')

## 4. Identify Economic Regimes

In [ ]:
# Classify each window into a regime based on volatility
vol_mean = tvp_volatilities.mean(axis=1)
vol_p33 = np.percentile(vol_mean, 33)
vol_p67 = np.percentile(vol_mean, 67)

regimes = []
for v in vol_mean:
    if v < vol_p33:
        regimes.append('expansion')
    elif v < vol_p67:
        regimes.append('slowdown')
    else:
        regimes.append('crisis')

# Compute regime-specific coefficient averages
regime_coefficients = {}
regime_volatilities = {}
for regime in ['expansion', 'slowdown', 'crisis']:
    mask = np.array([r == regime for r in regimes])
    if mask.sum() > 0:
        regime_coefficients[regime] = tvp_coefficients[mask].mean(axis=0)
        regime_volatilities[regime] = tvp_volatilities[mask].mean(axis=0)
        print(f'{regime}: {mask.sum()} windows, avg volatility: {vol_mean[mask].mean():.4f}')

print('\n✅ Regime-specific parameters computed')

## 5. Export for Local Inference

In [ ]:
import json

# Save regime parameters
output = {
    'variable_names': list(df.columns),
    'n_vars': n_vars,
    'regime_coefficients': {k: v.tolist() for k, v in regime_coefficients.items()},
    'regime_volatilities': {k: v.tolist() for k, v in regime_volatilities.items()},
    'baseline_stats': {
        'means': df.mean().to_dict(),
        'stds': df.std().to_dict(),
    },
}

with open('tvp_var_parameters.json', 'w') as f:
    json.dump(output, f, indent=2)

print('✅ Exported tvp_var_parameters.json')
print(f'   Variables: {output["variable_names"]}')
print(f'   Regimes: {list(regime_coefficients.keys())}')

## 6. Visualize Time-Varying Parameters

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Plot volatility over time with regime coloring
colors = {'expansion': '#34D399', 'slowdown': '#FBBF24', 'crisis': '#F87171'}
regime_colors = [colors[r] for r in regimes]

axes[0].bar(tvp_dates, vol_mean, color=regime_colors, width=90)
axes[0].set_title('Average Volatility Over Time (Regime Colored)', fontsize=14)
axes[0].set_ylabel('Avg Residual Variance')

# Plot first 3 coefficient evolution
for i in range(min(3, tvp_coefficients.shape[1])):
    axes[1].plot(tvp_dates, tvp_coefficients[:, i], label=f'Coef {i}', alpha=0.7)
axes[1].set_title('Sample Coefficient Evolution', fontsize=14)
axes[1].legend()

# Plot regime timeline
regime_map = {'expansion': 1, 'slowdown': 2, 'crisis': 3}
axes[2].plot(tvp_dates, [regime_map[r] for r in regimes], 'o-', markersize=3)
axes[2].set_yticks([1, 2, 3])
axes[2].set_yticklabels(['Expansion', 'Slowdown', 'Crisis'])
axes[2].set_title('Detected Economic Regime', fontsize=14)

plt.tight_layout()
plt.savefig('tvp_var_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved tvp_var_analysis.png')